# Online Advertising Performance Analysis
Company X • April–June 2020 • Python / VS Code

Run cells from top to bottom. First select your `.venv` kernel. This notebook calls the same functions as `analysis.py`; inspect that file to study their implementations. Open `outputs/report.html` for the full written answers.

**Goal:** answer 20 business questions and predict click counts. Revenue is income for the advertising firm and ad spend for Company X.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
from analysis import load_data, summarize, run

ROOT = Path.cwd()
assert (ROOT / "data/online_advertising_performance_data.csv").exists(), "Open this notebook from the project folder."
df, audit = load_data(ROOT / "data/online_advertising_performance_data.csv")
display(pd.Series(audit, name="Data quality"))
display(df.head())

## 1. Run the reproducible pipeline
This creates the tables, figures, report, and model evaluation. No online data is fetched. Duplicates and outliers are retained and flagged. Undefined rates are NaN. All rates are ratios of totals, not means of row-level rates.

In [ ]:
df, tables, model_metrics = run()
OUT = ROOT / "outputs"
def result(name):
    display(pd.read_csv(OUT / (name + ".csv")))

## Question 1: Overall engagement trend
Compare CTR by month and the impression-weighted mix of audience categories. Labels are targeting categories, not continuous engagement measurements.

In [ ]:
result("monthly")
result("monthly_engagement_impression_share")

## Question 2: Banner size and clicks
Compare raw clicks alongside CTR; unequal exposure affects click totals.

In [ ]:
result("banner")

## Question 3: Placements with the most displays and clicks
The anonymous placement codes cannot be mapped to publisher names or website/app types.

In [ ]:
result("placement")

## Question 4: Cost and revenue correlation
Pearson and Spearman measure association. Shared traffic volume can produce correlation.

In [ ]:
result("correlations")

## Question 5: Average revenue per click
Compute total revenue divided by total clicks. This is Company X spend per click, not Company X income.

In [ ]:
totals = df[["revenue", "clicks", "post_click_sales_amount"]].sum()
print("Company X spend per click: $", round(totals.revenue / totals.clicks, 4))
print("Attributed sales per click: $", round(totals.post_click_sales_amount / totals.clicks, 4))

## Question 6: Campaign conversion rates
Transactions divided by clicks can exceed 100% in some rows. This is not a unique-person conversion probability.

In [ ]:
result("campaign")

## Question 7: Sales over time
The seven-day mean helps reveal patterns. Sales are attributed to campaign dates, not necessarily transaction dates.

In [ ]:
result("monthly")

## Question 8: Engagement across banners
Compare the audience mix and within-category performance.

In [ ]:
result("banner_engagement_impression_share")
result("banner_engagement")

## Question 9: Placement conversion rates
Consider denominator size alongside the transaction-per-click rate.

In [ ]:
result("placement")

## Question 10: Seasonal or weekly fluctuations
Compare per-day volumes across weekdays. One quarter cannot establish annual seasonality.

In [ ]:
result("weekday")

## Question 11: Engagement and revenue
Use categorical comparisons and normalized metrics instead of arbitrary numeric encoding.

In [ ]:
result("engagement")

## Question 12: Outlier investigation
Global IQR flags are review prompts, not automatic deletion decisions.

In [ ]:
import json
print(json.loads((OUT / "outlier_thresholds.json").read_text()))
display(pd.read_csv(OUT / "outliers.csv").sort_values("cost", ascending=False).head(10))
result("duplicate_sensitivity")

## Question 13: Campaign × banner × placement
Inspect combinations with sufficient support, using an explicit heuristic threshold.

In [ ]:
combinations = tables["campaign_banner_placement"]
display(combinations.query("clicks >= 1000 and post_click_conversions >= 30").sort_values("company_x_roas", ascending=False).head(10))

## Question 14: ROI and consistent performance
Distinguish firm ROI from Company X ROAS. A monthly rank of 1 means the best observed result that month.

In [ ]:
result("campaign_monthly_consistency")
result("banner_monthly_consistency")

## Question 15: Conversion distribution by placement
Check total transactions and each placement’s share.

In [ ]:
result("placement")

## Question 16: Weekdays versus weekends
Use display-weighted category shares and per-day activity.

In [ ]:
result("day_type")
result("weekday_weekend_engagement_mix")

## Question 17: CPC by campaign and banner
Distinguish firm cost-per-click from Company X spend-per-click.

In [ ]:
result("campaign_banner")

## Question 18: Cost-effective conversion acquisition
Lower CPA describes lower attributed spend per transaction, not necessarily greater profit.

In [ ]:
result("campaign")
result("placement")

## Question 19: Conversion rates by weekday
These comparisons may be confounded by changing campaigns and audience mix.

In [ ]:
result("weekday")

## Question 20: Campaign effectiveness by audience category
Compare transaction rate, CPA and ROAS within each campaign.

In [ ]:
result("campaign_engagement")

## Visual summary
Daily trends, banner comparisons, placement performance, audience mix, and cost/weekday patterns.

In [ ]:
for name in ["01_daily_trends", "02_banner_performance", "03_placement_performance", "04_engagement", "05_cost_and_weekday"]:
    display(Image(filename=str(OUT / (name + ".png"))))

## Machine learning evaluation
Predict clicks **given impression volume**. Train on April; choose on May MAE; refit through May; test on June. Baseline: training CTR × displays. Candidates: Poisson gradient boosting. No cost, revenue, conversions, or sales features. See `train_models` in `analysis.py` for the complete implementation.

Compare error metrics and total predictions. A better MAE does not guarantee good total calibration; the included model substantially overpredicts June clicks.

In [ ]:
display(model_metrics)
display(Image(filename=str(OUT / "06_model_predictions.png")))

## Conclusions and limitations
Read the generated report for numerical answers. Explain the revenue/spend distinction, changing volume and CTR, confounding in banner comparisons, the 30-day attribution window, and incomplete knowledge of label maturity. The model is a learning baseline, not a budget optimizer. Request later data and validate tracking before recommending spend changes.